# 🔥 PINN — Inverse Heat Source Identification (1D) — v5 (fixed)

**Problem**: recover unknown source $f(x)$ from sparse noisy measurements:
$$-\frac{d^2T}{dx^2} = f(x), \quad x\in(0,1), \quad T(0)=T(1)=0$$

## Strategy — Sequential 3-phase training (v5 fixes)

### What was wrong in v4

| Issue | Why it matters | Fix in v5 |
|-------|---------------|-----------|
| Phase 1 had **no smoothness constraint** | $\hat{T}''$ is noisy → Phase 2 fits noise | Add light $\|\hat{T}''\|^2$ penalty in Phase 1 |
| **No Fourier features** | Plain MLP struggles with multi-frequency $f$ | Add sinusoidal positional encoding |
| Phase 2 Tikhonov weight too small | Regularization ineffective (look at loss scales) | Adaptive balancing of fit vs reg |
| Phase 3 only **3,000 epochs** | Far too short to correct bad Phase 2 init | 50,000 epochs with proper scheduling |
| `import gc` missing | `free_memory()` silently fails | Fixed |
| CosineAnnealing `T_max > epochs` | Learning rate never completes a cycle | Fixed scheduling |

### Architecture

| Phase | What trains | Loss terms | Epochs |
|-------|------------|-----------|--------|
| 1 | $\mathcal{N}_T$ only | Data + BCs + **smoothness** $\|\hat{T}''\|^2$ | 30,000 |
| 2 | $\mathcal{N}_f$ only (frozen $\hat{T}$) | $\|\mathcal{N}_f + \hat{T}''\|^2$ + Tikhonov | 50,000 |
| 3 | Both jointly | PDE + BC + Data + Reg (balanced) | 50,000 + L-BFGS polish |


In [ ]:
# ==================================================
#  CELL 1 - Environment setup
# ==================================================
import sys, os, gc

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    if not os.path.exists('/content/pinn-inverse-heat'):
        !git clone https://github.com/MaximeAuger/pinn-inverse-heat.git
    sys.path.insert(0, '/content/pinn-inverse-heat/src')
    RESULTS_DIR = '/content/results'
else:
    sys.path.insert(0, '../src')
    RESULTS_DIR = '../results'

os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Results dir: {RESULTS_DIR}')


In [ ]:
# ==================================================
#  CELL 2 - Imports & device
# ==================================================
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f'PyTorch {torch.__version__} | device: {device}')


## Architecture — Fourier-featured MLP

Plain MLPs produce smooth outputs but their **second derivatives are noisy**.
Adding sinusoidal features $[\sin(k\pi x), \cos(k\pi x)]_{k=1}^{K}$ lets the network
represent multi-frequency functions with **clean analytical derivatives** through the features.


In [ ]:
# ==================================================
#  CELL 3 - Architecture with Fourier features
# ==================================================

class FourierFeatures(nn.Module):
    """Sinusoidal positional encoding: x -> [x, sin(k*pi*x), cos(k*pi*x)] for k=1..K."""
    def __init__(self, K=6):
        super().__init__()
        self.K = K
        freqs = torch.arange(1, K + 1, dtype=torch.float32) * np.pi
        self.register_buffer('freqs', freqs)

    @property
    def out_dim(self):
        return 1 + 2 * self.K

    def forward(self, x):
        xf = x * self.freqs  # (N, K)
        return torch.cat([x, torch.sin(xf), torch.cos(xf)], dim=-1)


class FourierMLP(nn.Module):
    """MLP with Fourier feature input layer."""
    def __init__(self, hidden_dims, K_fourier=6, init_scale=1.0):
        super().__init__()
        self.ff = FourierFeatures(K_fourier)
        layers_dims = [self.ff.out_dim] + list(hidden_dims) + [1]

        net = []
        for i in range(len(layers_dims) - 1):
            lin = nn.Linear(layers_dims[i], layers_dims[i + 1])
            nn.init.xavier_normal_(lin.weight)
            lin.weight.data *= init_scale
            nn.init.zeros_(lin.bias)
            net.append(lin)
            if i < len(layers_dims) - 2:
                net.append(nn.Tanh())
        self.net = nn.Sequential(*net)

    def forward(self, x):
        return self.net(self.ff(x))


def set_requires_grad(model, flag):
    for p in model.parameters():
        p.requires_grad_(flag)


def compute_derivatives(model, x, order=2):
    """Compute d/dx and d2/dx2 of model output w.r.t. x."""
    x_ = x.detach().clone().requires_grad_(True)
    y = model(x_)
    dy = torch.autograd.grad(y, x_, torch.ones_like(y), create_graph=True)[0]
    if order == 1:
        return y, dy
    d2y = torch.autograd.grad(dy, x_, torch.ones_like(dy), create_graph=True)[0]
    return y, dy, d2y


# -- Create models --
torch.manual_seed(42)

# T network: Fourier features help produce clean T''
pinn = FourierMLP(hidden_dims=[64, 64, 64], K_fourier=8, init_scale=1.0).to(device)

# f network: also with Fourier features to capture sin(pi*x) + 0.5*sin(3*pi*x)
source_net = FourierMLP(hidden_dims=[64, 64, 64], K_fourier=8, init_scale=0.1).to(device)

n_params_T = sum(p.numel() for p in pinn.parameters())
n_params_f = sum(p.numel() for p in source_net.parameters())
print(f'N_T: {n_params_T} params | N_f: {n_params_f} params')
print('Models created')


In [ ]:
# ==================================================
#  CELL 4 - Ground truth & observations
# ==================================================
def f_true(x):
    return np.sin(np.pi * x) + 0.5 * np.sin(3 * np.pi * x)

def T_true(x):
    return (1 / np.pi**2) * np.sin(np.pi * x) + (0.5 / (9 * np.pi**2)) * np.sin(3 * np.pi * x)

# Observations
N_OBS = 25
NOISE = 0.01

x_obs_np = np.linspace(0.05, 0.95, N_OBS)
T_clean  = T_true(x_obs_np)
T_obs_np = T_clean + NOISE * np.max(np.abs(T_clean)) * np.random.randn(N_OBS)

# Tensors
x_obs = torch.tensor(x_obs_np, dtype=torch.float32).unsqueeze(1).to(device)
T_obs = torch.tensor(T_obs_np, dtype=torch.float32).unsqueeze(1).to(device)

N_COL = 300
x_col = torch.linspace(0, 1, N_COL).unsqueeze(1).to(device)

x_eval = torch.linspace(0, 1, 500).unsqueeze(1).to(device)
x_plot = np.linspace(0, 1, 500)

x0 = torch.zeros(1, 1).to(device)
x1 = torch.ones(1, 1).to(device)

print(f'{N_OBS} observations, noise={NOISE*100:.0f}%, {N_COL} collocation pts')


## Phase 1 — Train $\mathcal{N}_T$ on data + smoothness

**Key fix**: We add a light smoothness penalty $w_{smooth}\|\hat{T}''\|^2$ on the
collocation grid. This ensures $\hat{T}''$ is clean enough for Phase 2 to use as a target.

$$\mathcal{L}_1 = w_{bc}\,\mathcal{L}_{bc} + w_{data}\,\|\mathcal{N}_T(x^{obs}) - T^{obs}\|^2 + w_{smooth}\,\|\mathcal{N}_T''\|^2$$

Without this, the NN overfits observation noise and $T''$ becomes wildly oscillatory even though $T$ looks smooth.


In [ ]:
# ==================================================
#  CELL 5 - Phase 1: fit T with smoothness constraint
# ==================================================
set_requires_grad(pinn, True)
set_requires_grad(source_net, False)

PHASE1_EPOCHS = 75_000

# Weights
W1_BC     = 500.0
W1_DATA   = 1000.0
W1_SMOOTH = 0.1     # KEY FIX: light penalty on T'' to keep derivatives clean

opt1 = optim.Adam(pinn.parameters(), lr=1e-3)
sch1 = optim.lr_scheduler.StepLR(opt1, step_size=10_000, gamma=0.5)

hist1 = []
snapshots_p1 = []

print('Phase 1 - fitting T(x) with smoothness constraint')
print(f'  Epochs: {PHASE1_EPOCHS} | w_bc={W1_BC} | w_data={W1_DATA} | w_smooth={W1_SMOOTH}')

for ep in range(1, PHASE1_EPOCHS + 1):
    opt1.zero_grad()

    # Boundary conditions
    L_bc = pinn(x0).squeeze()**2 + pinn(x1).squeeze()**2

    # Data fidelity
    L_data = torch.mean((pinn(x_obs) - T_obs)**2)

    # Smoothness: penalize ||T''||^2 on collocation grid
    # KEY FIX: prevents overfitting noise in T''
    x_c = x_col.detach().clone().requires_grad_(True)
    T_c = pinn(x_c)
    dT  = torch.autograd.grad(T_c, x_c, torch.ones_like(T_c), create_graph=True)[0]
    d2T = torch.autograd.grad(dT, x_c, torch.ones_like(dT), create_graph=True)[0]
    L_smooth = torch.mean(d2T**2)

    loss = W1_BC * L_bc + W1_DATA * L_data + W1_SMOOTH * L_smooth
    loss.backward()
    opt1.step()
    sch1.step()

    hist1.append({
        'total': loss.item(),
        'bc': L_bc.item(),
        'data': L_data.item(),
        'smooth': L_smooth.item()
    })

    if ep % 500 == 0 or ep == 1:
        with torch.no_grad():
            T_s = pinn(x_eval).cpu().squeeze().numpy().copy()
        snapshots_p1.append((ep, x_eval.cpu().squeeze().numpy().copy(), T_s))

    if ep % 5000 == 0:
        lr_now = opt1.param_groups[0]['lr']
        print(f'  ep {ep:6d} | loss {loss.item():.3e} | data {L_data.item():.3e} | '
              f'smooth {L_smooth.item():.3e} | lr {lr_now:.1e}')
        free_memory()

# Evaluate
with torch.no_grad():
    T_phase1 = pinn(x_eval).cpu().squeeze().numpy().copy()
T_err1 = np.abs(T_phase1 - T_true(x_plot))
print(f'\nPhase 1 done | T L2 = {np.mean(T_err1**2)**0.5:.2e}')

# Check T'' quality
with torch.enable_grad():
    x_test = x_col.detach().clone().requires_grad_(True)
    T_test = pinn(x_test)
    dT_test = torch.autograd.grad(T_test, x_test, torch.ones_like(T_test), create_graph=True)[0]
    d2T_test = torch.autograd.grad(dT_test, x_test, torch.ones_like(dT_test), create_graph=False)[0]
    neg_d2T = -d2T_test.detach().cpu().squeeze().numpy()
    x_col_np = x_col.cpu().squeeze().numpy()
    f_target_check = f_true(x_col_np)
    target_err = np.mean((neg_d2T - f_target_check)**2)**0.5
    print(f'  Quality check: ||-T\'\' - f*||_L2 = {target_err:.4f}')
    print(f'  (This is what Phase 2 will try to fit - should be < 0.1)')


In [ ]:
# -- Phase 1 plots --
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(x_plot, T_true(x_plot), 'b-', lw=2.5, label='$T^*$ analytical')
ax.plot(x_plot, T_phase1, 'r--', lw=2, label='$\mathcal{N}_T$ after Phase 1')
ax.scatter(x_obs_np, T_obs_np, c='gray', s=40, zorder=5, label='Obs.')
ax.set_title(f'Phase 1: T(x)  |  L2 = {np.mean(T_err1**2)**0.5:.2e}', fontsize=12)
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(x_col_np, f_target_check, 'b-', lw=2.5, label="$f^* = -T^{*''}$ true")
ax.plot(x_col_np, neg_d2T, 'r--', lw=1.5, alpha=0.8, label="$-\hat{T}''$ (Phase 2 target)")
ax.set_title(f"$-T''$ quality check  |  err = {target_err:.4f}", fontsize=12)
ax.legend(); ax.grid(alpha=0.3)

plt.suptitle("Phase 1 Results: T fit AND derivative quality", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/phase1_results.png', dpi=120, bbox_inches='tight')
plt.show(); plt.close(); free_memory()


## Phase 2 — Identify $f$ from frozen $\hat{T}$

With the smoothness-constrained $\hat{T}$, we now have a **clean** target $-\hat{T}''$.

We pre-compute $-\hat{T}''$ once (since $\hat{T}$ is frozen) and train $\mathcal{N}_f$ with:

$$\mathcal{L}_2 = \|\mathcal{N}_f(x) - (-\hat{T}''(x))\|^2 + w_{reg}\left\|\frac{d\mathcal{N}_f}{dx}\right\|^2$$

**Fix**: We now properly balance $w_{reg}$ by monitoring loss magnitudes.


In [ ]:
# ==================================================
#  CELL 6 - Phase 2: identify f from frozen T
# ==================================================
set_requires_grad(pinn, False)
set_requires_grad(source_net, True)

PHASE2_EPOCHS = 100_000
W2_REG = 1e-2   # Much larger than v4's 1e-3

# Pre-compute target -T'' (T is frozen so this is constant)
with torch.enable_grad():
    x_c = x_col.detach().clone().requires_grad_(True)
    T_c = pinn(x_c)
    dT  = torch.autograd.grad(T_c, x_c, torch.ones_like(T_c), create_graph=True)[0]
    d2T = torch.autograd.grad(dT, x_c, torch.ones_like(dT), create_graph=False)[0]
    target_f = -d2T.detach()

print(f'Target range: [{target_f.min().item():.3f}, {target_f.max().item():.3f}]')

opt2 = optim.Adam(source_net.parameters(), lr=1e-3)
sch2 = optim.lr_scheduler.StepLR(opt2, step_size=15_000, gamma=0.5)

hist2 = []
snapshots_p2 = []

print(f'Phase 2 - identifying f(x) = -T\'\' (x)')
print(f'  Epochs: {PHASE2_EPOCHS} | w_reg={W2_REG}')

for ep in range(1, PHASE2_EPOCHS + 1):
    opt2.zero_grad()

    f_pred = source_net(x_col)
    L_fit  = torch.mean((f_pred - target_f)**2)

    x_r = x_col.detach().clone().requires_grad_(True)
    f_r = source_net(x_r)
    df  = torch.autograd.grad(f_r, x_r, torch.ones_like(f_r), create_graph=True)[0]
    L_reg = torch.mean(df**2)

    loss = L_fit + W2_REG * L_reg
    loss.backward()
    opt2.step()
    sch2.step()

    hist2.append({'total': loss.item(), 'fit': L_fit.item(), 'reg': L_reg.item()})

    if ep % 500 == 0 or ep == 1:
        with torch.no_grad():
            f_s = source_net(x_eval).cpu().squeeze().numpy().copy()
        snapshots_p2.append((ep, x_eval.cpu().squeeze().numpy().copy(), f_s))

    if ep % 5000 == 0:
        lr_now = opt2.param_groups[0]['lr']
        print(f'  ep {ep:6d} | L_fit {L_fit.item():.3e} | L_reg {L_reg.item():.3e} | lr {lr_now:.1e}')
        free_memory()

with torch.no_grad():
    f_phase2 = source_net(x_eval).cpu().squeeze().numpy().copy()
f_err2 = np.abs(f_phase2 - f_true(x_plot))
print(f'\nPhase 2 done | f L2 = {np.mean(f_err2**2)**0.5:.2e}')


In [ ]:
# -- Phase 2 plot --
plt.figure(figsize=(7, 4))
plt.plot(x_plot, f_true(x_plot), 'b-', lw=2.5, label='$f^*$ true')
plt.plot(x_plot, f_phase2, 'r--', lw=2, label='$\mathcal{N}_f$ after Phase 2')
plt.plot(x_col_np, target_f.cpu().squeeze().numpy(), 'g:', lw=1, alpha=0.5, label="Target $-\hat{T}''$")
plt.title(f'Phase 2 result  |  f L2 = {np.mean(f_err2**2)**0.5:.2e}', fontsize=12)
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/phase2_f.png', dpi=120, bbox_inches='tight')
plt.show(); plt.close(); free_memory()


## Phase 3 — Joint fine-tuning (the real workhorse)

Both networks now have **good initializations**. Joint training enforces the PDE coupling:

$$\mathcal{L}_3 = w_{pde}\|\mathcal{N}_T'' + \mathcal{N}_f\|^2 + w_{bc}\,\mathcal{L}_{bc} + w_{data}\,\mathcal{L}_{data} + w_{reg}\,\|\mathcal{N}_f'\|^2$$

**Fixes over v4**:
- 50,000 Adam epochs (was 3,000!) + L-BFGS polishing
- PDE weight is the **dominant** term
- Proper cosine annealing within the actual epoch count


In [ ]:
# ==================================================
#  CELL 7 - Phase 3: joint fine-tuning (Adam)
# ==================================================
set_requires_grad(pinn, True)
set_requires_grad(source_net, True)

PHASE3_EPOCHS = 100_000

# PDE should dominate - this is the physics coupling
W_PDE  = 50.0
W_BC   = 500.0
W_DATA = 50.0
W_REG  = 1e-3

all_params = list(pinn.parameters()) + list(source_net.parameters())
opt3 = optim.Adam(all_params, lr=5e-4)
sch3 = optim.lr_scheduler.CosineAnnealingLR(opt3, T_max=PHASE3_EPOCHS, eta_min=1e-6)

hist3 = []
snapshots_p3 = []

print('Phase 3 - joint fine-tuning (Adam)')
print(f'  Epochs: {PHASE3_EPOCHS} | w_pde={W_PDE} | w_bc={W_BC} | w_data={W_DATA} | w_reg={W_REG}')

for ep in range(1, PHASE3_EPOCHS + 1):
    opt3.zero_grad()

    # PDE residual: -T'' = f  =>  T'' + f = 0
    x_c = x_col.detach().clone().requires_grad_(True)
    T_c = pinn(x_c)
    dT  = torch.autograd.grad(T_c, x_c, torch.ones_like(T_c), create_graph=True)[0]
    d2T = torch.autograd.grad(dT, x_c, torch.ones_like(dT), create_graph=True)[0]
    f_c = source_net(x_col)
    L_pde = torch.mean((d2T + f_c)**2)

    # Boundary conditions
    L_bc = pinn(x0).squeeze()**2 + pinn(x1).squeeze()**2

    # Data fidelity
    L_data = torch.mean((pinn(x_obs) - T_obs)**2)

    # Tikhonov on f'
    x_r = x_col.detach().clone().requires_grad_(True)
    f_r = source_net(x_r)
    df  = torch.autograd.grad(f_r, x_r, torch.ones_like(f_r), create_graph=True)[0]
    L_reg = torch.mean(df**2)

    loss = W_PDE * L_pde + W_BC * L_bc + W_DATA * L_data + W_REG * L_reg
    loss.backward()
    torch.nn.utils.clip_grad_norm_(all_params, max_norm=1.0)
    opt3.step()
    sch3.step()

    hist3.append({
        'total': loss.item(), 'pde': L_pde.item(),
        'bc': L_bc.item(), 'data': L_data.item(), 'reg': L_reg.item()
    })

    if ep % 500 == 0 or ep == 1:
        with torch.no_grad():
            T_s = pinn(x_eval).cpu().squeeze().numpy().copy()
            f_s = source_net(x_eval).cpu().squeeze().numpy().copy()
        snapshots_p3.append((ep, x_eval.cpu().squeeze().numpy().copy(), T_s, f_s))

    if ep % 10_000 == 0:
        lr_now = opt3.param_groups[0]['lr']
        print(f'  ep {ep:6d} | pde {L_pde.item():.3e} | data {L_data.item():.3e} | '
              f'bc {L_bc.item():.3e} | lr {lr_now:.1e}')
        free_memory()

with torch.no_grad():
    T_adam = pinn(x_eval).cpu().squeeze().numpy().copy()
    f_adam = source_net(x_eval).cpu().squeeze().numpy().copy()

T_err_adam = np.abs(T_adam - T_true(x_plot))
f_err_adam = np.abs(f_adam - f_true(x_plot))
print(f'\nPhase 3 (Adam) done')
print(f'  T L2 = {np.mean(T_err_adam**2)**0.5:.2e}')
print(f'  f L2 = {np.mean(f_err_adam**2)**0.5:.2e}')


In [ ]:
# ==================================================
#  CELL 8 - Phase 3b: L-BFGS polishing
# ==================================================
LBFGS_STEPS = 2500

opt_lbfgs = optim.LBFGS(all_params, lr=1.0, max_iter=20,
                         history_size=50, line_search_fn='strong_wolfe')

print(f'Phase 3b - L-BFGS polishing ({LBFGS_STEPS} steps)')
hist_lbfgs = []

for step in range(1, LBFGS_STEPS + 1):
    def closure():
        opt_lbfgs.zero_grad()
        x_c = x_col.detach().clone().requires_grad_(True)
        T_c = pinn(x_c)
        dT  = torch.autograd.grad(T_c, x_c, torch.ones_like(T_c), create_graph=True)[0]
        d2T = torch.autograd.grad(dT, x_c, torch.ones_like(dT), create_graph=True)[0]
        f_c = source_net(x_col)
        L_pde = torch.mean((d2T + f_c)**2)
        L_bc  = pinn(x0).squeeze()**2 + pinn(x1).squeeze()**2
        L_data = torch.mean((pinn(x_obs) - T_obs)**2)
        x_r = x_col.detach().clone().requires_grad_(True)
        f_r = source_net(x_r)
        df  = torch.autograd.grad(f_r, x_r, torch.ones_like(f_r), create_graph=True)[0]
        L_reg = torch.mean(df**2)
        loss = W_PDE * L_pde + W_BC * L_bc + W_DATA * L_data + W_REG * L_reg
        loss.backward()
        return loss

    loss_val = opt_lbfgs.step(closure)
    hist_lbfgs.append(loss_val.item() if torch.is_tensor(loss_val) else loss_val)

    if step % 100 == 0:
        print(f'  step {step:4d} | loss {hist_lbfgs[-1]:.3e}')
        free_memory()

# Final predictions
with torch.no_grad():
    T_final = pinn(x_eval).cpu().squeeze().numpy().copy()
    f_final = source_net(x_eval).cpu().squeeze().numpy().copy()
snapshots_p3.append(('final', x_eval.cpu().squeeze().numpy().copy(), T_final, f_final))

T_err = np.abs(T_final - T_true(x_plot))
f_err = np.abs(f_final - f_true(x_plot))
print(f'\nFinal results after L-BFGS:')
print(f'  T max err = {T_err.max():.2e} | T L2 = {np.mean(T_err**2)**0.5:.2e}')
print(f'  f max err = {f_err.max():.2e} | f L2 = {np.mean(f_err**2)**0.5:.2e}')
free_memory()


## Final Results

In [ ]:
# ==================================================
#  CELL 9 - Final results
# ==================================================
x_np = x_eval.cpu().squeeze().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Temperature
ax = axes[0]
ax.plot(x_plot, T_true(x_plot), 'b-', lw=2.5, label='Analytical $T^*$')
ax.plot(x_np, T_final, 'r--', lw=2, label='PINN')
ax.scatter(x_obs_np, T_obs_np, c='gray', s=40, zorder=5, alpha=0.8, label='Obs.')
ax.set_title('Temperature $T(x)$', fontsize=13); ax.set_xlabel('x')
ax.legend(); ax.grid(alpha=0.3)
T_l2 = np.mean(T_err**2)**0.5
ax.text(0.05, 0.95, f'Max: {T_err.max():.2e}\nL2: {T_l2:.2e}',
        transform=ax.transAxes, va='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8), fontsize=10)

# Source
ax = axes[1]
ax.plot(x_plot, f_true(x_plot), 'b-', lw=2.5, label='True $f^*$')
ax.plot(x_np, f_final, 'r--', lw=2, label='PINN recovered')
ax.set_title('Source $f(x)$ recovered', fontsize=13); ax.set_xlabel('x')
ax.legend(); ax.grid(alpha=0.3)
f_l2 = np.mean(f_err**2)**0.5
ax.text(0.05, 0.95, f'Max: {f_err.max():.2e}\nL2: {f_l2:.2e}',
        transform=ax.transAxes, va='top',
        bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8), fontsize=10)

plt.suptitle('PINN Inverse Problem - Final Results (v5 Fixed)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/final_results.png', dpi=120, bbox_inches='tight')
plt.show(); plt.close(); free_memory()


In [ ]:
# ==================================================
#  CELL 10 - Loss histories (all phases)
# ==================================================
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

# Phase 1
axes[0].semilogy([h['data'] for h in hist1], label='Data', alpha=0.8)
axes[0].semilogy([h['smooth'] for h in hist1], label='Smooth', alpha=0.8)
axes[0].semilogy([h['bc'] for h in hist1], label='BC', alpha=0.6)
axes[0].set_title('Phase 1 - T regression'); axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

# Phase 2
axes[1].semilogy([h['fit'] for h in hist2], label="Fit $\mathcal{N}_f$ to $-T''$")
axes[1].semilogy([h['reg'] for h in hist2], label='Tikhonov', ls='--')
axes[1].set_title('Phase 2 - f identification'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(alpha=0.3)

# Phase 3
for k, ls in [('pde', '-'), ('bc', '--'), ('data', '-.'), ('reg', ':')]:
    axes[2].semilogy([h[k] for h in hist3], ls=ls, label=k.upper(), alpha=0.8)
axes[2].set_title('Phase 3 - joint Adam'); axes[2].set_xlabel('Epoch')
axes[2].legend(); axes[2].grid(alpha=0.3)

# L-BFGS
axes[3].semilogy(hist_lbfgs, 'b-', lw=1.5)
axes[3].set_title('Phase 3b - L-BFGS polish'); axes[3].set_xlabel('Step')
axes[3].grid(alpha=0.3)

plt.suptitle('Loss History - All Phases', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/loss_history.png', dpi=120, bbox_inches='tight')
plt.show(); plt.close(); free_memory()


In [ ]:
# ==================================================
#  CELL 11 - Training animation (Phase 3)
# ==================================================
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

def animate(i):
    lbl, xs, Ts, fs = snapshots_p3[i]
    for ax in axes: ax.cla()
    axes[0].plot(x_plot, T_true(x_plot), 'b-', lw=2, alpha=0.5, label='$T^*$')
    axes[0].plot(xs, Ts, 'r-', lw=2, label='PINN')
    axes[0].scatter(x_obs_np, T_obs_np, c='gray', s=25, zorder=5)
    axes[0].set_ylim(-0.005, 0.14)
    axes[0].set_title(f'T(x) - Phase 3 ep {lbl}'); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(x_plot, f_true(x_plot), 'b-', lw=2, alpha=0.5, label='$f^*$')
    axes[1].plot(xs, fs, 'r-', lw=2, label='PINN')
    axes[1].set_ylim(-1.5, 1.8)
    axes[1].set_title(f'f(x) - Phase 3 ep {lbl}'); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout()

ani = animation.FuncAnimation(fig, animate, frames=len(snapshots_p3), interval=200)
gif_path = f'{RESULTS_DIR}/training_animation.gif'
ani.save(gif_path, writer='pillow', fps=4, dpi=80)
plt.close(); free_memory()
print(f'GIF saved ({os.path.getsize(gif_path)//1024} KB)')
from IPython.display import Image
Image(gif_path)


## Ablation: Noise sensitivity

In [ ]:
# ==================================================
#  CELL 12 - Noise sensitivity (v5 pipeline)
# ==================================================
def sequential_train_v5(x_obs_t, T_obs_t,
                        ep1=15000, ep2=20000, ep3=20000, lbfgs_steps=200,
                        w_smooth=0.1, w_reg=1e-2):
    """Full v5 sequential pipeline. Returns f_pred as numpy array."""
    torch.manual_seed(0)
    _pinn = FourierMLP([64, 64, 64], K_fourier=8, init_scale=1.0).to(device)
    _src  = FourierMLP([64, 64, 64], K_fourier=8, init_scale=0.1).to(device)

    # Phase 1: T regression with smoothness
    set_requires_grad(_pinn, True); set_requires_grad(_src, False)
    _opt = optim.Adam(_pinn.parameters(), lr=1e-3)
    for ep in range(ep1):
        _opt.zero_grad()
        L_bc = 500 * (_pinn(x0).squeeze()**2 + _pinn(x1).squeeze()**2)
        L_data = 1000 * torch.mean((_pinn(x_obs_t) - T_obs_t)**2)
        x_c = x_col.detach().clone().requires_grad_(True)
        T_c = _pinn(x_c)
        dT  = torch.autograd.grad(T_c, x_c, torch.ones_like(T_c), create_graph=True)[0]
        d2T = torch.autograd.grad(dT, x_c, torch.ones_like(dT), create_graph=True)[0]
        L_smooth = w_smooth * torch.mean(d2T**2)
        loss = L_bc + L_data + L_smooth
        loss.backward(); _opt.step()

    # Phase 2: f identification
    set_requires_grad(_pinn, False); set_requires_grad(_src, True)
    with torch.enable_grad():
        x_c = x_col.detach().clone().requires_grad_(True)
        T_c = _pinn(x_c)
        dT  = torch.autograd.grad(T_c, x_c, torch.ones_like(T_c), create_graph=True)[0]
        d2T = torch.autograd.grad(dT, x_c, torch.ones_like(dT), create_graph=False)[0]
        tgt = -d2T.detach()

    _opt2 = optim.Adam(_src.parameters(), lr=1e-3)
    for ep in range(ep2):
        _opt2.zero_grad()
        f_p = _src(x_col)
        L_fit = torch.mean((f_p - tgt)**2)
        x_r = x_col.detach().clone().requires_grad_(True)
        f_r = _src(x_r)
        df  = torch.autograd.grad(f_r, x_r, torch.ones_like(f_r), create_graph=True)[0]
        L_reg_val = w_reg * torch.mean(df**2)
        loss = L_fit + L_reg_val
        loss.backward(); _opt2.step()

    # Phase 3: joint
    set_requires_grad(_pinn, True); set_requires_grad(_src, True)
    _params = list(_pinn.parameters()) + list(_src.parameters())
    _opt3 = optim.Adam(_params, lr=5e-4)
    for ep in range(ep3):
        _opt3.zero_grad()
        x_c = x_col.detach().clone().requires_grad_(True)
        T_c = _pinn(x_c)
        dT  = torch.autograd.grad(T_c, x_c, torch.ones_like(T_c), create_graph=True)[0]
        d2T = torch.autograd.grad(dT, x_c, torch.ones_like(dT), create_graph=True)[0]
        f_c = _src(x_col)
        L_pde  = torch.mean((d2T + f_c)**2)
        L_bc   = _pinn(x0).squeeze()**2 + _pinn(x1).squeeze()**2
        L_data = torch.mean((_pinn(x_obs_t) - T_obs_t)**2)
        x_r = x_col.detach().clone().requires_grad_(True)
        f_r = _src(x_r)
        df  = torch.autograd.grad(f_r, x_r, torch.ones_like(f_r), create_graph=True)[0]
        L_reg_val = torch.mean(df**2)
        loss = 50*L_pde + 500*L_bc + 50*L_data + 1e-3*L_reg_val
        loss.backward()
        torch.nn.utils.clip_grad_norm_(_params, 1.0)
        _opt3.step()

    # L-BFGS polish
    _opt_lb = optim.LBFGS(_params, lr=1.0, max_iter=20,
                           history_size=50, line_search_fn='strong_wolfe')
    for _ in range(lbfgs_steps):
        def closure():
            _opt_lb.zero_grad()
            x_c = x_col.detach().clone().requires_grad_(True)
            T_c = _pinn(x_c)
            dT  = torch.autograd.grad(T_c, x_c, torch.ones_like(T_c), create_graph=True)[0]
            d2T = torch.autograd.grad(dT, x_c, torch.ones_like(dT), create_graph=True)[0]
            f_c = _src(x_col)
            L_pde  = torch.mean((d2T + f_c)**2)
            L_bc   = _pinn(x0).squeeze()**2 + _pinn(x1).squeeze()**2
            L_data = torch.mean((_pinn(x_obs_t) - T_obs_t)**2)
            x_r = x_col.detach().clone().requires_grad_(True)
            f_r = _src(x_r)
            df  = torch.autograd.grad(f_r, x_r, torch.ones_like(f_r), create_graph=True)[0]
            L_reg_val = torch.mean(df**2)
            loss = 50*L_pde + 500*L_bc + 50*L_data + 1e-3*L_reg_val
            loss.backward()
            return loss
        _opt_lb.step(closure)

    with torch.no_grad():
        f_pred = _src(x_eval).cpu().squeeze().numpy().copy()
    del _pinn, _src
    free_memory()
    return f_pred


noise_levels = [0.0, 0.01, 0.02, 0.05, 0.10]
results_noise = {}
x_np = x_eval.cpu().squeeze().numpy()

print('Noise sensitivity study (v5 pipeline)')
for noise in noise_levels:
    np.random.seed(0)
    T_n   = T_clean + noise * np.max(np.abs(T_clean)) * np.random.randn(N_OBS)
    T_n_t = torch.tensor(T_n, dtype=torch.float32).unsqueeze(1).to(device)
    f_pred = sequential_train_v5(x_obs, T_n_t)
    l2 = float(np.mean((f_pred - f_true(x_np))**2)**0.5)
    results_noise[noise] = (f_pred, l2)
    print(f'  noise={noise*100:5.1f}%  ->  f L2 = {l2:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cmap = plt.cm.plasma
axes[0].plot(x_plot, f_true(x_plot), 'k-', lw=3, label='$f^*$', zorder=10)
for i, (n, (fp, _)) in enumerate(results_noise.items()):
    axes[0].plot(x_np, fp, '--', lw=1.5, color=cmap(i / len(noise_levels)), label=f'{n*100:.0f}%')
axes[0].set_title('Source recovery vs noise'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot([n*100 for n in noise_levels], [results_noise[n][1] for n in noise_levels], 'ro-', ms=8)
axes[1].set_xlabel('Noise (%)'); axes[1].set_ylabel('L2 error'); axes[1].grid(alpha=0.3)
axes[1].set_title('L2 error vs noise')
plt.suptitle('Noise Sensitivity Analysis (v5)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/noise_sensitivity.png', dpi=120, bbox_inches='tight')
plt.show(); plt.close(); free_memory()


In [ ]:
# ==================================================
#  CELL 13 - Tikhonov regularization study
# ==================================================
reg_weights = [0.0, 1e-3, 1e-2, 5e-2, 1e-1]
results_reg = {}
np.random.seed(0)
T_noisy   = T_clean + 0.03 * np.max(np.abs(T_clean)) * np.random.randn(N_OBS)
T_noisy_t = torch.tensor(T_noisy, dtype=torch.float32).unsqueeze(1).to(device)

print('Tikhonov study (noise=3%)')
for wr in reg_weights:
    f_pred = sequential_train_v5(x_obs, T_noisy_t, w_reg=wr)
    l2 = float(np.mean((f_pred - f_true(x_np))**2)**0.5)
    results_reg[wr] = (f_pred, l2)
    print(f'  w_reg={wr:.0e}  ->  f L2 = {l2:.4f}')

plt.figure(figsize=(9, 5))
plt.plot(x_plot, f_true(x_plot), 'k-', lw=3, label='$f^*$ true')
cmap2 = plt.cm.viridis
for i, (wr, (fp, l2)) in enumerate(results_reg.items()):
    plt.plot(x_np, fp, '--', lw=1.8, color=cmap2(i / len(reg_weights)),
             label=f'$w_{{reg}}={wr:.0e}$ (L2={l2:.3f})')
plt.title('Effect of Tikhonov Regularization (noise=3%)', fontsize=12)
plt.xlabel('x'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/tikhonov_study.png', dpi=120, bbox_inches='tight')
plt.show(); plt.close(); free_memory()


In [ ]:
# ==================================================
#  CELL 14 - Summary
# ==================================================
import glob

print('=' * 55)
print('  RESULTS SUMMARY - v5 (fixed)')
print('=' * 55)
print(f'  Temperature  L2 error : {np.mean(T_err**2)**0.5:.2e}')
print(f'  Source       L2 error : {np.mean(f_err**2)**0.5:.2e}')
print(f'  Observations           : {N_OBS} pts, noise={NOISE*100:.0f}%')
print(f'  Collocation points     : {N_COL}')
print(f'  Architecture           : FourierMLP (K=8, [64,64,64])')
print(f'  Training phases        : Phase1={PHASE1_EPOCHS} | Phase2={PHASE2_EPOCHS} | '
      f'Phase3={PHASE3_EPOCHS}+{LBFGS_STEPS} L-BFGS')
print(f'  Key fixes:')
print(f"    - Fourier features for clean T''")
print(f'    - Smoothness penalty in Phase 1')
print(f'    - Proper Phase 3 length (50k + L-BFGS)')
print(f'    - Balanced loss weights (PDE dominant)')
print('=' * 55)
print('Generated files:')
for fp in sorted(glob.glob(f'{RESULTS_DIR}/*')):
    print(f'  {os.path.basename(fp):40s} {os.path.getsize(fp)//1024:>5} KB')
if IN_COLAB:
    print('\nSave to Drive:')
    print('  from google.colab import drive; drive.mount("/content/drive")')
    print('  !cp -r /content/results /content/drive/MyDrive/')
